# 05 - Evaluation

Notebook đánh giá kết quả phân đoạn tổn thương da trên tập dữ liệu ISIC.

Các chỉ số sử dụng: Dice, IoU, Precision và Recall.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

## 1. Configuration

In [ ]:
GROUND_TRUTH_DIR = Path("../data/masks/test")
PREDICTION_DIR = Path("../results/segmentation/predictions")
RESULT_DIR = Path("../results/evaluation")

RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Ground Truth :", GROUND_TRUTH_DIR.resolve())
print("Prediction   :", PREDICTION_DIR.resolve())
print("Result       :", RESULT_DIR.resolve())

## 2. Load Binary Mask

Pixel có giá trị lớn hơn 0 được xem là vùng lesion.

In [ ]:
def load_mask(path):
    mask = Image.open(path).convert("L")
    mask = np.array(mask)
    return mask > 0

## 3. Calculate Metrics

TP: Ground Truth và Prediction đều là lesion.

FP: Prediction là lesion nhưng Ground Truth là background.

FN: Ground Truth là lesion nhưng Prediction là background.

In [ ]:
def calculate_metrics(ground_truth, prediction):
    ground_truth = ground_truth.astype(bool)
    prediction = prediction.astype(bool)

    tp = np.logical_and(
        ground_truth,
        prediction
    ).sum()

    fp = np.logical_and(
        ~ground_truth,
        prediction
    ).sum()

    fn = np.logical_and(
        ground_truth,
        ~prediction
    ).sum()

    dice = (2 * tp) / max(1, 2 * tp + fp + fn)
    iou = tp / max(1, tp + fp + fn)
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)

    return {
        "Dice": dice,
        "IoU": iou,
        "Precision": precision,
        "Recall": recall,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn)
    }

## 4. Find Matching Test Images

In [ ]:
mask_extensions = {".png", ".jpg", ".jpeg"}

ground_truth_files = sorted([
    file for file in GROUND_TRUTH_DIR.rglob("*")
    if file.is_file() and file.suffix.lower() in mask_extensions
])

evaluation_pairs = []

for ground_truth_path in ground_truth_files:
    prediction_path = PREDICTION_DIR / f"{ground_truth_path.stem}.png"

    if prediction_path.exists():
        evaluation_pairs.append(
            (ground_truth_path, prediction_path)
        )

print("Ground Truth files:", len(ground_truth_files))
print("Matched pairs     :", len(evaluation_pairs))
print("Missing predictions:", len(ground_truth_files) - len(evaluation_pairs))

## 5. Evaluate Test Set

In [ ]:
results = []

for ground_truth_path, prediction_path in evaluation_pairs:
    ground_truth = load_mask(ground_truth_path)
    prediction = load_mask(prediction_path)

    if ground_truth.shape != prediction.shape:
        print(
            f"Shape mismatch: {ground_truth_path.name} "
            f"GT={ground_truth.shape}, "
            f"Prediction={prediction.shape}"
        )
        continue

    metrics = calculate_metrics(
        ground_truth,
        prediction
    )

    metrics["Image"] = ground_truth_path.stem
    results.append(metrics)

results_df = pd.DataFrame(results)

if len(results_df) > 0:
    results_df = results_df[
        [
            "Image",
            "Dice",
            "IoU",
            "Precision",
            "Recall",
            "TP",
            "FP",
            "FN"
        ]
    ]

print("Evaluated images:", len(results_df))
results_df.head()

## 6. Average Metrics

In [ ]:
if len(results_df) > 0:
    mean_dice = results_df["Dice"].mean()
    mean_iou = results_df["IoU"].mean()
    mean_precision = results_df["Precision"].mean()
    mean_recall = results_df["Recall"].mean()

    print("=" * 50)
    print("AVERAGE EVALUATION RESULTS")
    print("=" * 50)
    print(f"Mean Dice      : {mean_dice:.4f}")
    print(f"Mean IoU       : {mean_iou:.4f}")
    print(f"Mean Precision : {mean_precision:.4f}")
    print(f"Mean Recall    : {mean_recall:.4f}")
    print("=" * 50)
else:
    print("Chưa có dữ liệu để đánh giá.")

## 7. Display Evaluation Table

In [ ]:
if len(results_df) > 0:
    display(
        results_df.style.format({
            "Dice": "{:.4f}",
            "IoU": "{:.4f}",
            "Precision": "{:.4f}",
            "Recall": "{:.4f}"
        })
    )
else:
    print("Không có kết quả để hiển thị.")

## 8. Visualize Best and Worst Results

In [ ]:
if len(results_df) > 0:
    best_image = results_df.loc[
        results_df["Dice"].idxmax(),
        "Image"
    ]

    worst_image = results_df.loc[
        results_df["Dice"].idxmin(),
        "Image"
    ]

    print("Best result :", best_image)
    print("Worst result:", worst_image)
else:
    best_image = None
    worst_image = None
    print("Không có dữ liệu.")

In [ ]:
def show_result(image_name):
    ground_truth_path = GROUND_TRUTH_DIR / f"{image_name}.png"
    prediction_path = PREDICTION_DIR / f"{image_name}.png"

    if not ground_truth_path.exists():
        print("Không tìm thấy Ground Truth:", ground_truth_path)
        return

    if not prediction_path.exists():
        print("Không tìm thấy Prediction:", prediction_path)
        return

    ground_truth = load_mask(ground_truth_path)
    prediction = load_mask(prediction_path)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(ground_truth, cmap="gray")
    axes[0].set_title("Ground Truth")
    axes[0].axis("off")

    axes[1].imshow(prediction, cmap="gray")
    axes[1].set_title("Prediction")
    axes[1].axis("off")

    axes[2].imshow(ground_truth, cmap="gray")
    axes[2].imshow(prediction, alpha=0.4, cmap="Reds")
    axes[2].set_title("Comparison")
    axes[2].axis("off")

    plt.suptitle(image_name)
    plt.tight_layout()
    plt.show()

In [ ]:
if best_image is not None:
    print("BEST RESULT")
    show_result(best_image)

if worst_image is not None:
    print("WORST RESULT")
    show_result(worst_image)

## 9. Plot Metric Distribution

In [ ]:
if len(results_df) > 0:
    metrics = [
        "Dice",
        "IoU",
        "Precision",
        "Recall"
    ]

    means = [results_df[m].mean() for m in metrics]

    plt.figure(figsize=(8, 5))
    plt.bar(metrics, means)
    plt.ylim(0, 1)
    plt.ylabel("Score")
    plt.title("Average Segmentation Metrics")
    plt.grid(axis="y")
    plt.show()
else:
    print("Không có dữ liệu để vẽ biểu đồ.")

## 10. Save Evaluation Results

In [ ]:
csv_path = RESULT_DIR / "segmentation_evaluation.csv"

if len(results_df) > 0:
    results_df.to_csv(
        csv_path,
        index=False
    )

    print("Saved:", csv_path)
else:
    print("Không có kết quả để lưu.")

## 11. Final Evaluation Summary

Kết quả đánh giá được sử dụng để xác định khả năng phân đoạn vùng tổn thương của mô hình U-Net.

Dice và IoU phản ánh mức độ chồng lấp giữa Ground Truth và Prediction. Precision phản ánh mức độ chính xác của vùng được dự đoán là lesion. Recall phản ánh khả năng phát hiện đầy đủ vùng lesion.

In [ ]:
print("=" * 60)
print("ISIC SKIN LESION SEGMENTATION - EVALUATION")
print("=" * 60)
print("Number of evaluated images:", len(results_df))

if len(results_df) > 0:
    print(f"Mean Dice      : {results_df['Dice'].mean():.4f}")
    print(f"Mean IoU       : {results_df['IoU'].mean():.4f}")
    print(f"Mean Precision : {results_df['Precision'].mean():.4f}")
    print(f"Mean Recall    : {results_df['Recall'].mean():.4f}")
    print("CSV result     :", csv_path)
else:
    print("Chưa có prediction để đánh giá.")

print("=" * 60)